# 네이버 뉴스 기사 주소 수집 — 검색어 방식

네이버 뉴스 **검색창에 검색어를 넣은 결과**에서 기사 주소를 모으는 노트북임.
크롬 창을 띄워(Selenium) 하루치 검색 결과를 끝까지 스크롤한 뒤, 기사 주소만 골라 JSON으로 저장함.

**쓰는 순서**
1. 아래 `설정` 셀에서 `query_ranges`(검색어 + 기간)만 바꾸기
2. 위에서부터 셀 순서대로 실행
3. 저장 폴더에 `링크_{검색어}_{기간}.json`이 생기면 끝 → `검색어_뉴스_본문_수집.ipynb`로 넘어감

- 입력: `query_ranges`의 검색어, 시작일, 종료일
- 출력: `링크_{검색어}_{YYMMDD}_{YYMMDD}.json`
- 보조 파일: 수집로그 JSON(날짜별 건수), 중간 저장 JSON, 실패 목록 JSON
- 로컬/Colab 어느 쪽에서 열어도 알아서 경로를 잡음. 중간에 끊겨도 중간 저장 파일이 있으면 이어서 수집

> 개인 학습·연구용으로 쓰는 걸 전제로 함. 대기 시간을 줄이거나 여러 개를 동시에 돌리는 식으로 서버에 부담 주지 말 것


In [ ]:
# 필요한 도구 설치 — 이미 깔려 있으면 건너뛰어도 됨
# %pip install -q selenium beautifulsoup4 pandas

# Colab에서 돌릴 때만 — 크롬이 없어서 따로 설치해야 함 (로컬은 크롬만 깔려 있으면 됨)
# !apt-get -qq update && apt-get -qq install -y chromium-browser chromium-chromedriver

In [ ]:
# ============================== 설정 ==============================
# 여기만 고치면 됨. 아래 셀들은 그대로 실행
# =================================================================

# 검색어와 기간 — 검색어 하나당 시작일 ~ 종료일
# 날짜 형식: 'YYYY.MM.DD', 줄을 늘리면 여러 검색어를 이어서 수집함
query_ranges = [
    {'query': '기후변화', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
    # {'query': '전기차 보조금', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
]

# 기간이 길 때 월 단위로 잘라서 파일을 나눌지 여부
# False면 기간 전체가 파일 1개, True면 달마다 파일 1개 (몇 달치를 돌릴 때 중간 재개가 편함)
SPLIT_BY_MONTH = False

# 크롬 창을 보면서 돌리려면 False, 창 없이 돌리려면 True
# Colab은 화면이 없어서 자동으로 True가 됨
HEADLESS = False

# 저장 폴더 — 결과 파일이 전부 여기에 쌓임
# 직접 정하려면 경로 문자열 입력 (예: '/home/me/뉴스수집', Colab이면 '/content/drive/MyDrive/내프로젝트/data')
# None이면 자동 — 로컬은 프로젝트 폴더 아래 data/뉴스수집, Colab은 드라이브의 MyDrive/data/뉴스수집
# 드라이브에 따로 만들어 둔 프로젝트 폴더 안에 넣고 싶으면 그 경로를 직접 적을 것
SAVE_DIR_OVERRIDE = None

# Colab에서 구글 드라이브에 저장할지 여부 — True면 드라이브 연결 후 MyDrive 아래에 저장
MOUNT_DRIVE = True

# 최종 링크 파일이 이미 있는 기간은 다시 수집하지 않음
SKIP_COMPLETED = True

# 서버에 부담 주지 않도록 쉬는 시간(초) — 너무 줄이면 차단될 수 있음
DAY_PAUSE_RANGE_SEC = (2, 5)    # 날짜 하나 끝내고 다음 날짜로 넘어갈 때
JOB_PAUSE_RANGE_SEC = (10, 25)  # 검색어(작업) 하나 끝내고 다음 작업으로 넘어갈 때
SCROLL_PAUSE_SEC = 1.5          # 스크롤 한 번 내리고 새 기사가 뜰 때까지 기다리는 시간

# 검색 결과 정렬 — '최신순' / '오래된순' / '정확도순' 중 하나
# 하루치를 통째로 긁어서 결과 자체는 같지만, 창을 보면서 확인할 때 순서가 달라짐
SORT_ORDER = '최신순'

In [ ]:
import calendar
import json
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import unicodedata
import urllib.parse
from datetime import datetime, timedelta
from pathlib import Path

# 지금 Colab에서 돌고 있는지 확인 — 경로와 창 설정이 달라짐
IN_COLAB = 'google.colab' in sys.modules

# Colab은 화면이 없어서 크롬 창을 띄울 수 없음
if IN_COLAB and not HEADLESS:
    HEADLESS = True
    print('Colab이라 창 없이 실행함 (HEADLESS=True)')

# Colab이면 드라이브 연결 — 연결해야 수집 결과가 세션 종료 후에도 남음
if IN_COLAB and MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'드라이브 연결 실패 — 세션 안에만 저장됨: {exc!r}')


# 저장 폴더의 기준이 될 위치 찾기
# Colab은 드라이브 최상단(MyDrive), 로컬은 현재 폴더부터 위로 올라가며 프로젝트 최상위(.git 등)를 찾음
# Colab에는 프로젝트 폴더라는 게 따로 없어서 MyDrive 기준임 — 다른 곳에 넣으려면 위 SAVE_DIR_OVERRIDE 사용
def detect_project_dir():
    if IN_COLAB:
        drive_root = Path('/content/drive/MyDrive')
        return drive_root if drive_root.exists() else Path('/content')

    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / '.git').exists() or (candidate / 'pipeline_py').exists():
            return candidate
    # 못 찾으면 지금 폴더를 그대로 씀
    return here


PROJECT_DIR = detect_project_dir()

# 위 설정대로 저장 폴더 결정 — 링크 파일, 수집로그, 중간 저장 파일, 실패 목록이 모두 여기에 쌓임
SAVE_DIR = Path(SAVE_DIR_OVERRIDE) if SAVE_DIR_OVERRIDE else PROJECT_DIR / 'data' / '뉴스수집'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 네이버에 접속할 때 쓰는 기본 정보 — 흔한 크롬 브라우저와 같은 값
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'


# 검색어를 파일 이름으로 쓸 수 있게 정리
# 'LG U+' -> 'LG_U+', '주식/투자' -> '주식_투자' 처럼 공백과 못 쓰는 글자만 바꿈
def safe_name(text):
    name = unicodedata.normalize('NFC', str(text)).strip()
    name = re.sub(r'\s+', '_', name)
    name = re.sub(r'[\\/:*?"<>|]', '_', name)
    return name or 'query'


# 파일 이름에 붙는 기간 표시 만들기 (2026.05.05 ~ 2026.05.11 -> '260505_260511')
def make_period_suffix(start_date, end_date):
    return f"{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}"


# 설정한 검색어·기간을 실제로 돌릴 작업 목록으로 바꿈
# SPLIT_BY_MONTH가 True면 달 단위로 쪼개되, 시작·끝은 설정한 기간을 넘지 않게 자름
def build_jobs(query_ranges, split_by_month=SPLIT_BY_MONTH):
    jobs = []
    for item in query_ranges:
        query = item['query']
        start = datetime.strptime(item['start_date'], '%Y.%m.%d')
        end = datetime.strptime(item['end_date'], '%Y.%m.%d')
        # 시작일이 종료일보다 늦으면 기간을 잘못 적은 것이므로 바로 알려 줌
        if start > end:
            raise ValueError(f'시작일이 종료일보다 늦습니다: {item}')

        if not split_by_month:
            jobs.append({
                'query': query,
                'start_date': start.strftime('%Y.%m.%d'),
                'end_date': end.strftime('%Y.%m.%d'),
            })
            continue

        current = start
        while current <= end:
            # 그 달의 마지막 날 자동 계산 (2월 29일 같은 것도 알아서 맞음)
            last_day = calendar.monthrange(current.year, current.month)[1]
            month_end = current.replace(day=last_day)
            chunk_end = min(month_end, end)
            jobs.append({
                'query': query,
                'start_date': current.strftime('%Y.%m.%d'),
                'end_date': chunk_end.strftime('%Y.%m.%d'),
            })
            current = month_end + timedelta(days=1)  # 다음 달 1일로 이동
    return jobs


jobs = build_jobs(query_ranges)

print(f'실행 환경: {"Colab" if IN_COLAB else "로컬"}')
print(f'기준 폴더: {PROJECT_DIR}')
print(f'저장 폴더: {SAVE_DIR}')
print(f'크롬 창 보임: {not HEADLESS}')
print(f'총 작업 수: {len(jobs)}')
for job in jobs:
    print('  ', job, '-> 링크_' + safe_name(job['query']) + '_' + make_period_suffix(job['start_date'], job['end_date']) + '.json')

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By


# 이 컴퓨터에 깔린 크롬(또는 크로미움) 위치 찾기
# 위치를 직접 알려 주면 실행 오류가 줄어듦
def find_chrome_binary():
    candidates = []

    if platform.system() == 'Windows':
        # 윈도우에서 크롬이 자주 설치되는 폴더들을 후보로 둠
        candidates.extend([
            os.path.expandvars(r'%ProgramFiles%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%ProgramFiles(x86)%\Google\Chrome\Application\chrome.exe'),
            os.path.expandvars(r'%LocalAppData%\Google\Chrome\Application\chrome.exe'),
        ])
    else:
        # 맥·리눅스·WSL·Colab에서는 이름 후보를 차례로 확인
        for name in ['google-chrome', 'google-chrome-stable', 'chromium-browser', 'chromium']:
            found = shutil.which(name)
            if found:
                candidates.append(found)
        # 맥에서 앱으로 설치한 경우
        candidates.append('/Applications/Google Chrome.app/Contents/MacOS/Google Chrome')

    for path in candidates:
        if path and Path(path).exists():
            return str(Path(path))
    return None


# 검색 결과 페이지를 열 크롬 준비
def build_driver(headless=None):
    headless = HEADLESS if headless is None else headless

    options = Options()
    options.add_argument(f'user-agent={USER_AGENT}')  # 접속 정보를 일정하게 유지
    options.add_argument('--lang=ko-KR')  # 한국어 페이지로 받기
    options.add_experimental_option('excludeSwitches', ['enable-automation'])  # 자동 실행 창처럼 보이는 표시 줄이기
    options.add_experimental_option('useAutomationExtension', False)  # 자동 실행 창처럼 보이는 표시 줄이기
    options.add_argument('--disable-blink-features=AutomationControlled')  # 자동 실행 창처럼 보이는 표시 줄이기
    options.add_argument('--window-size=1400,1000')  # 항상 비슷한 화면 크기로 열기

    if headless:
        options.add_argument('--headless=new')  # 창 없이 실행

    if platform.system() != 'Windows':
        options.add_argument('--no-sandbox')  # 리눅스/WSL/Colab에서 크롬 실행 오류 줄이기
        options.add_argument('--disable-dev-shm-usage')  # 크롬이 중간에 꺼지는 문제 줄이기
        options.add_argument('--disable-gpu')  # 창 없이 실행할 때 그래픽 관련 오류 줄이기

    # 크롬 실행 파일을 찾으면 직접 지정, 못 찾으면 Selenium이 알아서 찾게 둠
    chrome_binary = find_chrome_binary()
    if chrome_binary:
        options.binary_location = chrome_binary
        print(f'크롬 위치: {chrome_binary}')
    else:
        print('크롬을 직접 찾지 못해 Selenium 기본 방식으로 실행 — 오류가 나면 크롬 설치 여부부터 확인')

    # 크롬 버전에 맞는 실행 도구는 Selenium이 알아서 받아 옴
    driver = webdriver.Chrome(service=Service(), options=options)

    # 네이버가 자동 실행 창이라고 판단할 가능성을 조금 낮춤
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
    })
    return driver


driver = build_driver()
print('크롬 준비 완료')

In [ ]:
# 랜덤 대기 후 로그 출력 — 매번 똑같은 간격으로 요청하지 않게 함
def polite_sleep(label, pause_range):
    pause_sec = random.uniform(*pause_range)
    print(f'{label} {pause_sec:.1f}초 대기')
    time.sleep(pause_sec)


# 하루치 검색 결과 주소 만들기
# 네이버 뉴스 검색은 한 번에 너무 많은 기간을 넣으면 뒤쪽 결과가 잘려서 하루 단위로 끊어 검색함
# 네이버가 쓰는 정렬 번호 — 설정에 적은 한글을 번호로 바꿔 줌
SORT_CODES = {'정확도순': '0', '최신순': '1', '오래된순': '2'}


def build_search_url(query, day_str, sort_order=SORT_ORDER):
    encoded_query = urllib.parse.quote(query)
    ymd = day_str.replace('.', '')
    # nso 값에는 ':'와 ','가 들어가서 주소용으로 한 번 더 변환해야 함
    nso_value = f'so:r,p:from{ymd}to{ymd}'
    sort_code = SORT_CODES.get(sort_order, '1')
    return (
        f'https://search.naver.com/search.naver?ssc=tab.news.all'
        f'&query={encoded_query}&sm=tab_opt&sort={sort_code}&photo=0&field=0'
        f'&pd=3&ds={day_str}&de={day_str}&docid=&related=0&mynews=0'
        f'&office_type=0&office_section_code=0&news_office_checked='
        f'&nso={urllib.parse.quote(nso_value)}&is_sug_officeid=0'
        f'&office_category=&service_area=2'
    )


# 검색 결과는 아래로 내릴수록 기사가 더 붙는 방식이라 끝까지 스크롤해야 전부 보임
# 더 이상 길어지지 않으면 한 번 더 기다렸다가 확인하고 끝냄
def scroll_to_bottom(driver, pause_sec=SCROLL_PAUSE_SEC):
    last_height = driver.execute_script('return document.body.scrollHeight')
    scroll_count = 0

    while True:
        scroll_count += 1
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(pause_sec)

        new_height = driver.execute_script('return document.body.scrollHeight')
        if new_height == last_height:
            # 인터넷이 느려서 늦게 붙는 경우가 있어 한 번 더 기다린 뒤 확인
            time.sleep(2.0)
            final_height = driver.execute_script('return document.body.scrollHeight')
            if final_height == new_height:
                break  # 두 번 확인해도 그대로면 진짜 끝
            last_height = final_height
            continue
        last_height = new_height

    return scroll_count


# 화면에 뜬 링크 중 네이버 뉴스 기사 주소만 골라냄
# 주소 뒤에 붙는 부가 정보는 떼어 내서 같은 기사를 두 번 세지 않게 함
ARTICLE_LINK_PATTERN = re.compile(r'^https://n\.news\.naver\.com/(?:mnews/)?article/\d+/\d+')


def extract_article_links(driver):
    links = set()
    for a in driver.find_elements(By.XPATH, '//a[contains(@href, "n.news.naver.com")]'):
        href = a.get_attribute('href') or ''
        if ARTICLE_LINK_PATTERN.match(href):
            links.add(href.split('?')[0])
    return links


# 검색어 하나 × 기간 하나를 통합 JSON 1개로 저장
# 날짜별로 중간 저장하기 때문에 중간에 끊겨도 다음 실행 때 이어서 함
def collect_links(query, start_date, end_date, driver=None, save_dir=SAVE_DIR):
    # 크롬 창을 따로 넘기지 않으면 위 셀에서 만들어 둔 driver 사용
    if driver is None:
        driver = globals()['driver']

    name = safe_name(query)
    period = make_period_suffix(start_date, end_date)

    # temp는 중간 저장, links는 최종 결과, stats는 날짜별 수집 기록
    temp_links_path = save_dir / f'중간저장_{name}_{period}.json'
    links_save_path = save_dir / f'링크_{name}_{period}.json'
    stats_save_path = save_dir / f'수집로그_{name}_{period}.json'

    # 최종 파일이 이미 있으면 같은 작업은 다시 하지 않음
    if SKIP_COMPLETED and links_save_path.exists():
        print()
        print(f'=== {query} / {start_date} ~ {end_date} 이미 완료됨, 건너뜀 ===')
        print(f'기존 파일: {links_save_path}')
        return links_save_path

    print()
    print(f'=== {query} / {start_date} ~ {end_date} 수집 시작 ===')

    # 중간 저장 파일이 있으면 이어서 — last_date 다음 날부터 수집
    if temp_links_path.exists():
        with temp_links_path.open('r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        all_links_set = set(checkpoint.get('links', []))
        last_collected_date = checkpoint.get('last_date')
        daily_stats = checkpoint.get('daily_stats', [])
        print(f'중간 저장 파일에서 링크 {len(all_links_set)}개 불러옴 — {last_collected_date} 다음부터 이어서 수집')
    else:
        all_links_set = set()
        last_collected_date = None
        daily_stats = []
        print('새로 수집 시작')

    current = datetime.strptime(start_date, '%Y.%m.%d')
    end = datetime.strptime(end_date, '%Y.%m.%d')

    while current <= end:
        day_str = current.strftime('%Y.%m.%d')

        # 이미 끝낸 날짜는 건너뜀
        if last_collected_date and day_str <= last_collected_date:
            print(f'{day_str} — 이미 수집 완료, 건너뜀')
            current += timedelta(days=1)
            continue

        started_at = time.time()
        driver.get(build_search_url(query, day_str))

        scroll_count = scroll_to_bottom(driver)
        day_links = extract_article_links(driver)

        before = len(all_links_set)
        all_links_set.update(day_links)
        added = len(all_links_set) - before  # 날짜끼리 겹치는 기사를 뺀 실제 증가분
        elapsed = round(time.time() - started_at, 2)

        # 날짜별 기록 — 유독 적게 잡힌 날이 있으면 나중에 확인할 때 씀
        daily_stats.append({
            'date': day_str,
            'found': len(day_links),
            'added': added,
            'total': len(all_links_set),
            'scroll_count': scroll_count,
            'elapsed_sec': elapsed,
        })

        print(
            f'{day_str} — {len(day_links)}건 수집 / 신규 {added}건 추가 '
            f'/ 누적 {len(all_links_set)}건 / 스크롤 {scroll_count}회 / {elapsed}초'
        )

        # 하루 끝날 때마다 중간 저장 — 중간에 끊겨도 여기까지는 남음
        with temp_links_path.open('w', encoding='utf-8') as f:
            json.dump(
                {'query': query, 'links': sorted(all_links_set), 'last_date': day_str, 'daily_stats': daily_stats},
                f, ensure_ascii=False, indent=2,
            )
        last_collected_date = day_str
        current += timedelta(days=1)
        # 마지막 날 뒤에는 쉬지 않음
        if current <= end:
            polite_sleep('다음 날짜 전', DAY_PAUSE_RANGE_SEC)

    # 기간 전체 링크를 하나로 저장 — 순서를 정리해 실행할 때마다 파일이 뒤집히지 않게 함
    naver_news_links = sorted(all_links_set)
    with links_save_path.open('w', encoding='utf-8') as f:
        json.dump(naver_news_links, f, ensure_ascii=False, indent=2)

    with stats_save_path.open('w', encoding='utf-8') as f:
        json.dump({
            'query': query,
            'start_date': start_date,
            'end_date': end_date,
            'total': len(naver_news_links),
            'days': daily_stats,
        }, f, ensure_ascii=False, indent=2)

    # 정상적으로 끝났으면 중간 저장 파일은 지움
    if temp_links_path.exists():
        temp_links_path.unlink()

    print(f'수집 완료 — 총 {len(naver_news_links)}개')
    print(f'링크 저장: {links_save_path}')
    print(f'수집 로그 저장: {stats_save_path}')
    return links_save_path


# 작업 목록을 순서대로 실행 (작업 하나 = 검색어 × 기간)
# 하나가 실패해도 기록만 남기고 다음 작업으로 넘어감
results = []
failures = []
for index, job in enumerate(jobs, start=1):
    print()
    print(f'[{index}/{len(jobs)}] 작업 실행: {job}')
    try:
        results.append(collect_links(driver=driver, **job))
    except Exception as exc:
        failures.append({'job': job, 'error': repr(exc)})
        print(f'작업 실패, 다음 작업으로 넘어감: {exc!r}')
    finally:
        if index < len(jobs):
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

# 실패한 작업이 있으면 나중에 다시 돌릴 수 있게 파일로 남김
if failures:
    failures_path = SAVE_DIR / '수집실패목록_url.json'
    with failures_path.open('w', encoding='utf-8') as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)
    print()
    print(f'실패 작업 {len(failures)}개 저장: {failures_path}')

print()
print('전체 작업 완료')
print(f'성공/건너뜀: {len(results)}개, 실패: {len(failures)}개')
for result_path in results:
    print(result_path)

In [ ]:
# 작업이 끝나면 크롬 창 종료
# 필요하면 이 셀만 따로 실행
try:
    driver.quit()
    print('브라우저 종료 완료')
except Exception as exc:
    print(f'브라우저 종료 중 오류: {exc!r}')